In [ ]:
import os
import sys
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

sys.path.append('../src')
from config import *
from prepare_data import load_data
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, category in enumerate(CATEGORIES):
    images = os.listdir(os.path.join(INPUT_DIR, category))[:5]
    for j, img_name in enumerate(images):
        img = imread(os.path.join(INPUT_DIR, category, img_name))
        axes[i, j].imshow(img)
        axes[i, j].set_title(category)
        axes[i, j].axis('off')
plt.suptitle('Sample Images', fontsize=16)
plt.tight_layout()
plt.show()
counts = []
for category in CATEGORIES:
    count = len(os.listdir(os.path.join(INPUT_DIR, category)))
    counts.append(count)
    print('{}: {} images'.format(category, count))

plt.figure(figsize=(8, 5))
plt.bar(CATEGORIES, counts, color=['green', 'red'])
plt.title('Class Distribution')
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.show()
data, labels = load_data()
print('Total samples:', len(data))
print('Image features:', data.shape[1])

x_train, x_test, y_train, y_test = train_test_split(
    data, labels, test_size=TEST_SIZE, shuffle=True, stratify=labels
)
print('Train samples:', len(x_train))
print('Test samples:', len(x_test))
model = pickle.load(open('../models/model.p', 'rb'))

y_prediction = model.predict(x_test)
y_train_prediction = model.predict(x_train)

print('Train Accuracy: {}%'.format(round(
    (y_train_prediction == y_train).mean() * 100, 2)))
print('Test Accuracy: {}%'.format(round(
    (y_prediction == y_test).mean() * 100, 2)))
print('Classification Report:')
print(classification_report(y_test, y_prediction, target_names=CATEGORIES))
cm = confusion_matrix(y_test, y_prediction)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
test_images = []
test_labels = []

for i, category in enumerate(CATEGORIES):
    images = os.listdir(os.path.join(INPUT_DIR, category))
    random_images = random.sample(images, 5)  # randomly pick 5
    for img_name in random_images:
        img = imread(os.path.join(INPUT_DIR, category, img_name))
        test_images.append(img)
        test_labels.append(i)

# shuffle combined list together
combined = list(zip(test_images, test_labels))
random.shuffle(combined)
test_images, test_labels = zip(*combined)

for idx, ax in enumerate(axes.flatten()):
    img = test_images[idx]
    img_resized = resize(img, IMAGE_SIZE) / 255.0
    prediction = model.predict(img_resized.flatten().reshape(1, -1))
    ax.imshow(img)
    ax.set_title('Pred: {}\nActual: {}'.format(
        CATEGORIES[prediction[0]],
        CATEGORIES[test_labels[idx]]))
    ax.axis('off')

plt.suptitle('Sample Predictions', fontsize=16)
plt.tight_layout()
plt.show()